In [1]:
import tensorflow as tf
import os

In [2]:
model = tf.keras.models.load_model(
    "mobilenet_preprocessed.keras"
)

In [3]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("mobilenet_quantized.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\iamsh\AppData\Local\Temp\tmp0o96v300\assets


INFO:tensorflow:Assets written to: C:\Users\iamsh\AppData\Local\Temp\tmp0o96v300\assets


Saved artifact at 'C:\Users\iamsh\AppData\Local\Temp\tmp0o96v300'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1327901194768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327901195920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327901195344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327901195728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327901195536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919596176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919596944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919596560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919596752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919597520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327

In [4]:
baseline_size = os.path.getsize(
    "mobilenet_preprocessed.keras"
)/(1024*1024)

tflite_size = os.path.getsize(
    "mobilenet_quantized.tflite"
)/(1024*1024)

print("Baseline:", round(baseline_size,2),"MB")
print("TFLite:", round(tflite_size,2),"MB")

Baseline: 11.06 MB
TFLite: 9.1 MB


In [5]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_tflite_model = converter.convert()

with open("mobilenet_dynamic_quant.tflite", "wb") as f:
    f.write(quantized_tflite_model)

INFO:tensorflow:Assets written to: C:\Users\iamsh\AppData\Local\Temp\tmp0gc6whkq\assets


INFO:tensorflow:Assets written to: C:\Users\iamsh\AppData\Local\Temp\tmp0gc6whkq\assets


Saved artifact at 'C:\Users\iamsh\AppData\Local\Temp\tmp0gc6whkq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1327901194768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327901195920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327901195344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327901195728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327901195536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919596176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919596944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919596560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919596752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327919597520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1327

In [6]:
quantized_size = os.path.getsize(
    "mobilenet_dynamic_quant.tflite"
)/(1024*1024)

print("Quantized Size:", round(quantized_size,2), "MB")

Quantized Size: 2.57 MB


In [1]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(
    model_path="mobilenet_dynamic_quant.tflite"
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

c:\Users\iamsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [3]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

dataset_path = "Data/dataset-resized"

img_size = (224, 224)
batch_size = 32

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = val_ds.map(
    lambda x, y: (preprocess_input(x), y)
)

Found 2527 files belonging to 6 classes.
Using 505 files for validation.


In [4]:
print(input_details)

[{'name': 'serving_default_input_layer_1:0', 'index': 0, 'shape': array([  1, 224, 224,   3], dtype=int32), 'shape_signature': array([ -1, 224, 224,   3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]


In [5]:
correct = 0
total = 0

for images, labels in val_ds:

    images = images.numpy()
    labels = labels.numpy()

    for i in range(len(images)):

        input_data = np.expand_dims(
            images[i].astype(np.float32),
            axis=0
        )

        interpreter.set_tensor(
            input_details[0]["index"],
            input_data
        )

        interpreter.invoke()

        prediction = interpreter.get_tensor(
            output_details[0]["index"]
        )

        predicted_class = np.argmax(prediction)

        if predicted_class == labels[i]:
            correct += 1

        total += 1

accuracy = correct / total

print(f"Quantized Model Accuracy: {accuracy*100:.2f}%")

Quantized Model Accuracy: 85.15%


In [6]:
import tensorflow as tf
import numpy as np
import os

model = tf.keras.models.load_model(
    "mobilenet_preprocessed.keras"
)

In [7]:
def representative_dataset():

    for images, labels in train_ds.take(100):

        for i in range(len(images)):
            yield [images[i:i+1]]

In [9]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

with open(
    "mobilenet_full_integer_quant.tflite",
    "wb"
) as f:
    f.write(tflite_quant_model)

INFO:tensorflow:Assets written to: C:\Users\iamsh\AppData\Local\Temp\tmp9o8o7tbw\assets


INFO:tensorflow:Assets written to: C:\Users\iamsh\AppData\Local\Temp\tmp9o8o7tbw\assets


Saved artifact at 'C:\Users\iamsh\AppData\Local\Temp\tmp9o8o7tbw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1650920787216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920789136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920788944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920789328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920788752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920787408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920790288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920789712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920790480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920790864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650

c:\Users\iamsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


NameError: name 'train_ds' is not defined

In [11]:
import os

print(os.listdir())

['.git', '.gitignore', '.venv', 'baseline_model.keras', 'cnn_v2_baseline.keras', 'cnn_v4_batchnorm.keras', 'Data', 'green_ai_code.ipynb', 'LICENSE', 'mobilenet_baseline.keras', 'mobilenet_dynamic_quant.tflite', 'mobilenet_preprocessed.keras', 'mobilenet_quantized.tflite', 'mobilenet_transfer_learning.ipynb', 'mobilenet_transfer_learning2.ipynb', 'quantization.ipynb']


In [13]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

dataset_path = "Data/dataset-resized"

img_size = (224,224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y)
)

Found 2527 files belonging to 6 classes.
Using 2022 files for training.


In [14]:
def representative_dataset():
    for images, labels in train_ds.take(100):
        for i in range(len(images)):
            yield [images[i:i+1]]

In [15]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

print("Conversion completed!")

with open("mobilenet_full_integer_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)

print("File saved!")

INFO:tensorflow:Assets written to: C:\Users\iamsh\AppData\Local\Temp\tmp0baiwy98\assets


INFO:tensorflow:Assets written to: C:\Users\iamsh\AppData\Local\Temp\tmp0baiwy98\assets


Saved artifact at 'C:\Users\iamsh\AppData\Local\Temp\tmp0baiwy98'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1650920787216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920789136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920788944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920789328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920788752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920787408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920790288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920789712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920790480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650920790864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1650

c:\Users\iamsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Conversion completed!
File saved!


In [16]:
import os

size_mb = os.path.getsize(
    "mobilenet_full_integer_quant.tflite"
)/(1024*1024)

print(f"Full INT8 Size: {size_mb:.2f} MB")

Full INT8 Size: 2.76 MB


In [17]:
interpreter = tf.lite.Interpreter(
    model_path="mobilenet_full_integer_quant.tflite"
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(input_details)

[{'name': 'serving_default_input_layer_1:0', 'index': 0, 'shape': array([  1, 224, 224,   3], dtype=int32), 'shape_signature': array([ -1, 224, 224,   3], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.007843137718737125, -1), 'quantization_parameters': {'scales': array([0.00784314], dtype=float32), 'zero_points': array([-1], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]


c:\Users\iamsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [18]:
import numpy as np

correct = 0
total = 0

input_scale, input_zero_point = input_details[0]["quantization"]

for images, labels in val_ds:

    images = images.numpy()
    labels = labels.numpy()

    for i in range(len(images)):

        image = images[i:i+1]

        image_int8 = image / input_scale + input_zero_point
        image_int8 = np.clip(image_int8, -128, 127).astype(np.int8)

        interpreter.set_tensor(
            input_details[0]["index"],
            image_int8
        )

        interpreter.invoke()

        output = interpreter.get_tensor(
            output_details[0]["index"]
        )

        predicted_class = np.argmax(output)

        if predicted_class == labels[i]:
            correct += 1

        total += 1

accuracy = correct / total

print(f"Full INT8 Accuracy: {accuracy*100:.2f}%")

Full INT8 Accuracy: 83.96%
